In [1]:
import requests

In [25]:
endpoint_url = "https://query.wikidata.org/sparql"
headers = {
    "Accept": "application/sparql-results+json"
}

query = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?event ?eventLabel ?date WHERE {
  SERVICE <https://query.wikidata.org/sparql> {
    ?event wdt:P585 ?date .
    
    FILTER(?date >= "1933-01-02T00:00:00Z"^^xsd:dateTime &&
           ?date <= "1933-12-31T23:59:59Z"^^xsd:dateTime)
           
    OPTIONAL {
      ?event rdfs:label ?eventLabel .
      FILTER(LANG(?eventLabel) = "de" || LANG(?eventLabel) = "en")
    }
  }
}
ORDER BY ?date
LIMIT 1000
"""

query = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?event ?eventLabel ?date WHERE {
  SERVICE <https://query.wikidata.org/sparql> {
    ?event wdt:P585 ?date .
    
    FILTER(?date >= "1938-01-02T00:00:00Z"^^xsd:dateTime &&
           ?date <= "1938-12-31T23:59:59Z"^^xsd:dateTime)

    # Ensure it's a historical event or subclass
    ?event wdt:P31/wdt:P279* wd:Q1656682 .
    
    # Filter for events in Germany and the German Reich
    FILTER(?country IN (wd:Q7318, wd:Q28108, wd:Q16957, wd:Q1198, wd:Q183, wd:Q1206012)) .
    ?event wdt:P17 ?country .


    OPTIONAL {
      ?event rdfs:label ?eventLabel .
      FILTER(LANG(?eventLabel) = "de" || LANG(?eventLabel) = "en")
    }
  }
}
ORDER BY ?date
LIMIT 100
"""

In [26]:
response = requests.get(endpoint_url, params={"query": query}, headers=headers)
if response.status_code != 200:
    print(f"Error {response.status_code}: {response.text}")
else:
    data = response.json()

    for item in data["results"]["bindings"]:
        print(item["event"]["value"], item.get("eventLabel", {}).get("value", ""), item["date"]["value"])


http://www.wikidata.org/entity/Q28154369 1937 Tschammerpokal Final 1938-01-09T00:00:00Z
http://www.wikidata.org/entity/Q670166 Eiskunstlauf-Weltmeisterschaft 1938 1938-02-01T00:00:00Z
http://www.wikidata.org/entity/Q670166 1938 World Figure Skating Championships 1938-02-01T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q1532978 Septemberverschwörung 1938-09-01T00:00:00Z
http://www.wikidata.org/entity/Q1532978 Oster Conspiracy 1938-0

In [14]:
len(data["results"]["bindings"])

800